In [1]:
import numpy as np
import pandas as pd
import re
from sklearn.neighbors import BallTree

## Inspect and clean FAULTS dataset

In [2]:
faults_df = pd.read_csv(filepath_or_buffer="../data/J1939Faults.csv", low_memory=False)

In [3]:
faults_df.sample(5)

,RecordID,ESS_Id,EventTimeStamp,eventDescription,actionDescription,ecuSoftwareVersion,ecuSerialNumber,ecuModel,ecuMake,ecuSource,spn,fmi,active,activeTransitionCount,faultValue,EquipmentID,MCTNumber,Latitude,Longitude,LocationTimeStamp
824034,845398,22967031,2017-08-08 13:30:12.000,Abnormal Update Rate Tire Location,NaN,NaN,NaN,CECU3B-NAMUX4,PACCR,49,929,9,False,126,NaN,1646,105420184,35.989212,-86.571990,2017-08-08 13:30:08.000
346674,352554,6960232,2016-01-20 08:21:34.000,Abnormal Update Rate Tire Location,NaN,unknown,unknown,unknown,unknown,49,929,9,False,126,NaN,1623,105338710,33.563101,-86.831574,2016-01-20 08:21:29.000
1030095,1069996,65828963,2018-10-22 16:47:05.000,Low (Severity Medium) Engine Coolant Level,NaN,04358814*06087682*061516161145*09401661*G1*BDR*,79908054,6X1u13D1500000000,CMMNS,0,111,18,False,126,NaN,1894,105303702,37.087268,-80.622916,2018-10-22 16:47:00.000
175233,177619,4208306,2015-08-22 16:29:51.000,Low (Severity Low) Engine Coolant Level,NaN,unknown,unknown,unknown,unknown,0,111,17,False,1,NaN,1392,105420811,34.855972,-81.005648,2015-08-22 16:29:32.000
1148641,1204869,105466053,2019-10-09 19:32:33.000,Low (Severity Medium) Engine Coolant Level,NaN,04384413*22012490*031617122339*60701702*G1*BGT*,79993827,6X1u17D1500000000,CMMNS,0,111,18,False,6,NaN,2073,105309860,39.012407,-78.345138,2019-10-09 19:32:28.000


In [4]:
faults_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1187335 entries, 0 to 1187334
Data columns (total 20 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   RecordID               1187335 non-null  int64  
 1   ESS_Id                 1187335 non-null  int64  
 2   EventTimeStamp         1187335 non-null  object 
 3   eventDescription       1126490 non-null  object 
 4   actionDescription      0 non-null        float64
 5   ecuSoftwareVersion     891285 non-null   object 
 6   ecuSerialNumber        844318 non-null   object 
 7   ecuModel               1122577 non-null  object 
 8   ecuMake                1122577 non-null  object 
 9   ecuSource              1187335 non-null  int64  
 10  spn                    1187335 non-null  int64  
 11  fmi                    1187335 non-null  int64  
 12  active                 1187335 non-null  bool   
 13  activeTransitionCount  1187335 non-null  int64  
 14  faultValue        

### Possible Features:
* RecordID is unique
* ESS_Id all nan
* EventTimeStamp has 1,050,909 unique values (no nan)
* eventDescription has 60,845 nan (Create 'severity' column based on descriptions)
* actionDescription all nan
* ecuSoftwareVersion has 1,899 unique values (296,050 nan)
* ecuSerialNumber all nan
* ecuModel has 30 unique values (64,758 nan)
* ecuMake has 23 unique values (64,758 nan)
* ecuSource has 5 unique values (no nan)
* spn has 450 unique values (no nan)
* fmi has 26 unique values (no nan)
* active has 2 unique values (no nan)
* activeTransitionCount has 8,128 unique values (no nan)
* faultValue all nan
* EquipmentID has 1,927 unique values (no nan)
* MCTNumber has 768 unique values (no nan)
* Latitude has 211,823 unique values (no nan)
* Longitude has 265,211 unique values (no nan)
* LocationTimeStamp has 1,036,006 unique values (no nan)

In [5]:
# Convert timestamps to datetime objects
faults_df['EventTimeStamp'] = pd.to_datetime(faults_df['EventTimeStamp'])
faults_df['LocationTimeStamp'] = pd.to_datetime(faults_df['LocationTimeStamp'])
print(f'EventTimeStamp datatype: {faults_df['LocationTimeStamp'].dtype}')

EventTimeStamp datatype: datetime64[ns]


In [6]:
# Convert SPN and FMI values to strings
faults_df['spn'] = faults_df['spn'].astype(str)
faults_df['fmi'] = faults_df['fmi'].astype(str)
print(f'spn datatype: {faults_df['spn'].dtype}')

spn datatype: object


In [7]:
# Create column for active codes that are near service stations
service_stations = [
    (36.0666667, -86.4347222),
    (35.5883333, -86.4438888),
    (36.1950, -83.174722)
]

earth_radius_km = 6371.0
    
station_radians = np.radians(service_stations)
points_radians = np.radians(faults_df[['Latitude', 'Longitude']].values)
    
tree = BallTree(station_radians, metric='haversine')
    
# Query radius in radians
indices = tree.query_radius(X=points_radians, r=1.0 / earth_radius_km)
    
faults_df['NearServiceStation'] = np.array([len(idx) > 0 for idx in indices])
faults_df['NearServiceStation'].value_counts()

NearServiceStation
False    1055968
True      131367
Name: count, dtype: int64

In [8]:
# Create column for full derate flag
# Full derate is determined by spn code 5246 and engine code active is true
faults_df['IsFullDerate'] = (
        (faults_df['spn'] == '5246')
        & faults_df['active']
        & ~faults_df['NearServiceStation']
    )
faults_df['IsFullDerate'].isna().sum()

np.int64(0)

In [9]:
# Create columns for severity level from eventDescription
def extract_severity(text):

    if pd.isna(text):
        return np.nan

    # Severity with "Low", "Medium", or "high"
    pattern = r'Severity\s+(Low|Medium|High)'

    # Find pattern
    match = re.search(pattern, text)

    if match: 
        return f"Severity {match.group(1)}"
    else: 
        return np.nan

faults_df['Severity_Level'] = faults_df['eventDescription'].apply(extract_severity)

severity_map = {
    'Severity Low': 1,
    'Severity Medium': 2,
    'Severity High': 3
}

faults_df['Severity_Level_Numeric'] = faults_df['Severity_Level'].map(severity_map)

In [10]:
# Create target columns for full derate windows
def create_target_column(window_max:float, window_min:float, df:'DataFrame'=faults_df) -> None:

    df = df.sort_values(['EquipmentID', 'EventTimeStamp'])

    column_name = f'Derate_Target'
    
    # Initialize target column
    faults_df[column_name] = 0

    # Dataframe with just the derate events
    derate_events_df = faults_df[faults_df['IsFullDerate']].copy()

    # Group by EquipmentID 
    for equipment_id, group in faults_df.groupby('EquipmentID'):
        # Get derate events for this truck only
        truck_derates = derate_events_df[derate_events_df['EquipmentID'] == equipment_id]
        
        if len(truck_derates) > 0:
            # Get indices and timestamps for this truck's rows
            truck_indices = group.index
            truck_timestamps = group['EventTimeStamp'].values
            
            # For each derate event in this truck
            for _, derate_row in truck_derates.iterrows():
                derate_time = derate_row['EventTimeStamp']
                
                # Window: 
                window_start = derate_time - pd.Timedelta(hours=window_max)
                window_end = derate_time - pd.Timedelta(hours=window_min)

                # All events in the prediction window
                in_window = (truck_timestamps >= window_start) & (truck_timestamps <= window_end)
                target_indices_to_mark = truck_indices[in_window]

                # All events within and including the derate
                imminent_derate = (truck_timestamps > window_end) & (truck_timestamps <= derate_time)
                derate_indices_to_mark = truck_indices[imminent_derate]
                
                # Marked as predicting a derate
                faults_df.loc[target_indices_to_mark, column_name] = 1
                faults_df.loc[derate_indices_to_mark, column_name] = 2                

    print(f"Total events: {len(faults_df)}")
    print(f"Other: {(faults_df[column_name] == 0).sum()}")
    print(f"Events 2-8 hrs prior to derate: {(faults_df[column_name] == 1).sum()}")
    print(f"Events 2 hrs prior to and including derate: {(faults_df[column_name] == 2).sum()}")

In [11]:
create_target_column(window_max=8.0, window_min=2.0)

Total events: 1187335
Other: 1185150
Events 2-8 hrs prior to derate: 1085
Events 2 hrs prior to and including derate: 1100


In [12]:
faults_df['Derate_Target'].isna().sum()

np.int64(0)

## Inspect DIAGNOSTICS

In [13]:
diagnostics_df = pd.read_csv(filepath_or_buffer="../data/VehicleDiagnosticOnboardData.csv", low_memory=False)
diagnostics_df

,Id,Name,Value,FaultId
0,1,IgnStatus,False,1
1,2,EngineOilPressure,0,1
2,3,EngineOilTemperature,96.74375,1
3,4,TurboBoostPressure,0,1
4,5,EngineLoad,11,1
...,...,...,...,...
12821621,12864020,EngineCoolantTemperature,181.4,1248457
12821622,12864021,ParkingBrake,False,1248457
12821623,12864022,SwitchedBatteryVoltage,14.1,1248457
12821624,12864023,DistanceLtd,28606.65625,1248457


In [14]:
diagnostics_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12821626 entries, 0 to 12821625
Data columns (total 4 columns):
 #   Column   Dtype 
---  ------   ----- 
 0   Id       int64 
 1   Name     object
 2   Value    object
 3   FaultId  int64 
dtypes: int64(2), object(2)
memory usage: 391.3+ MB


In [15]:
# To get the on-board diagnostics at the time of the fault code, we can match the **RecordID** to the **FaultId**.
diagnostics_df.loc[diagnostics_df['FaultId'] == 1]

,Id,Name,Value,FaultId
0,1,IgnStatus,False,1
1,2,EngineOilPressure,0,1
2,3,EngineOilTemperature,96.74375,1
3,4,TurboBoostPressure,0,1
4,5,EngineLoad,11,1
5,6,AcceleratorPedal,0,1
6,7,IntakeManifoldTemperature,78.8,1
7,8,FuelRate,0,1
8,9,FuelLtd,12300.907429328,1
9,10,EngineRpm,0,1


## Merge FAULTS and DIAGNOSTICS

### Diagnostics 1-to-1 with faults

In [16]:
# Pivot DIAGNOSTICS wider
diagnostics_pivot_wider_df = diagnostics_df.pivot(
    columns='Name',
    index='FaultId',
    values='Value'
)
diagnostics_pivot_wider_df.sample(5)

Name,AcceleratorPedal,BarometricPressure,CruiseControlActive,CruiseControlSetSpeed,DistanceLtd,EngineCoolantTemperature,EngineLoad,EngineOilPressure,EngineOilTemperature,EngineRpm,...,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure
FaultId,,,,,,,,,,,,,,,,,,,,,
662257,72.4,14.2825,False,64.6226,482812.7,186.8,67,32.48,208.7937,1231.875,...,NaN,True,80.6,1023,NaN,NaN,57.75839,NaN,NaN,14.5
580671,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1023,NaN,NaN,NaN,NaN,NaN,NaN
841206,0,14.645,False,66.48672,465647.6,98.6,15,40.6,92.4125,599.375,...,NaN,True,89.6,1279,False,NaN,0,NaN,100,0.29
1224890,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1023,NaN,NaN,NaN,NaN,NaN,NaN
177100,94,14.2825,False,64.6226,598.483,177.8,98,41.18,208.7375,1260.375,...,32,True,95,1023,NaN,NaN,62.32159,NaN,100,29.29


In [17]:
# Replace commas with decimal points and convert to floats
float_columns = [
    'AcceleratorPedal',
    'BarometricPressure',
    'DistanceLtd',
    'EngineCoolantTemperature',
    'EngineOilPressure',
    'EngineOilTemperature',
    'EngineRpm',
    'EngineTimeLtd',
    'FuelLevel',
    'FuelLtd',
    'FuelRate',
    'FuelTemperature',
    'IntakeManifoldTemperature',
    'Speed',
    'SwitchedBatteryVoltage',
    'Throttle',
    'TurboBoostPressure'
]

for col in float_columns:
    print(col)
    diagnostics_pivot_wider_df[col] = diagnostics_pivot_wider_df[col].str.replace(pat=',', repl='.').astype(float)

AcceleratorPedal
BarometricPressure
DistanceLtd
EngineCoolantTemperature
EngineOilPressure
EngineOilTemperature
EngineRpm
EngineTimeLtd
FuelLevel
FuelLtd
FuelRate
FuelTemperature
IntakeManifoldTemperature
Speed
SwitchedBatteryVoltage
Throttle
TurboBoostPressure


In [18]:
faults_diagnostics_df = pd.merge(
    left=faults_df,
    right=diagnostics_pivot_wider_df,
    how='inner',
    left_on='RecordID',
    right_on='FaultId',
    validate='1:1'
)
faults_diagnostics_df['Derate_Target'].isna().sum()

np.int64(0)

In [19]:
faults_diagnostics_df.to_csv('../data/faults_diagnostics.csv', index=False)

In [20]:
# Split data into training and testing
cutoff_date = '2018-12-31 23:59:59'

training_faults_diagnostics_df = faults_diagnostics_df[faults_diagnostics_df['EventTimeStamp'] <= cutoff_date].reset_index()
testing_faults_diagnostics_df = faults_diagnostics_df[faults_diagnostics_df['EventTimeStamp'] > cutoff_date].reset_index()

In [21]:
training_faults_diagnostics_df.shape

(1058069, 50)

In [22]:
testing_faults_diagnostics_df.shape

(129266, 50)

In [23]:
training_faults_diagnostics_df.to_csv('../data/training_faults_diagnostics.csv', index=False)
testing_faults_diagnostics_df.to_csv('../data/testing_faults_diagnostics.csv', index=False)